# March Madness LSTM Recurrent Model

This notebook builds an LSTM-based recurrent neural network to predict game outcomes using:
1. **Team Encoder**: LSTM to encode each team's historical stats to a latent vector
2. **Prediction Head**: Binary classifier combining two team embeddings
3. **Walk-Forward Training**: Temporal validation preventing future leakage

## Setup and Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Libraries imported")
print(f"  Device: {device}")

✓ Libraries imported
  Device: cuda


## Hyperparameters

In [2]:
# Model hyperparameters (easily adjustable)
CONFIG = {
    # Model architecture
    'latent_dim': 32,              # Dimensionality of team latent vector
    'hidden_dim': 64,             # LSTM hidden dimension
    'num_layers': 3,               # Number of LSTM layers
    'dropout': 0.5,                # Dropout rate
    'use_team_embedding': False,   # Include learnable team embeddings
    'team_embedding_dim': 16,      # Dimension of team embeddings (if used)
    
    # Training
    'batch_size': 32,
    'learning_rate': 1e-3,
    'epochs': 10,
}


## 1. Load and Prepare Data

In [3]:
# Load processed data
df = pd.read_csv("./InputData2023.csv")
df.columns

Index(['Team 1', 'Team 2', 'Date', 'Site', 'Outcome', '1-ORtg', '1-DRtg', '1-3PAr', '1-TS%', '1-TRB%', '1-AST%', '1-STL%', '1-BLK%', '1-eFG%', '1-TOV%', '1-ORB%', '1-FTr', '1-oeFG%', '1-oTOV%', '1-oDRB%', '1-oFTr', '1-Pace', '1-o3PAr', '2-ORtg', '2-DRtg', '2-3PAr', '2-TS%', '2-TRB%', '2-AST%', '2-STL%', '2-BLK%', '2-eFG%', '2-TOV%', '2-ORB%', '2-FTr', '2-oeFG%', '2-oTOV%', '2-oDRB%', '2-oFTr', '2-Pace', '2-o3PAr', 'Game ID', 'is_tournament_game'], dtype='object')

## 2. Team Encoder LSTM Model

In [4]:
class TeamEncoder(nn.Module):
    """Encodes a team's game history to a latent vector using LSTM."""
    
    def __init__(self, input_size, hidden_dim, num_layers, latent_dim, dropout=0.1):
        super(TeamEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.latent_dim = latent_dim
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Project LSTM output to latent space
        self.projection = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_length, input_size) - sequence of game stats
        Returns:
            latent: (batch_size, latent_dim) - team encoding
        """
        # LSTM forward pass
        _, (h_n, _) = self.lstm(x)  # h_n: (num_layers, batch_size, hidden_dim)
        
        # Take last layer's hidden state
        last_hidden = h_n[-1]  # (batch_size, hidden_dim)
        
        # Project to latent dimension
        latent = self.projection(last_hidden)  # (batch_size, latent_dim)
        
        return latent


class PredictionHead(nn.Module):
    """Binary classifier head that combines two team embeddings."""
    
    def __init__(self, latent_dim, hidden_dim=64):
        super(PredictionHead, self).__init__()
        
        # Take difference between teams as input
        self.fc1 = nn.Linear(latent_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, team_a_latent, team_b_latent):
        """
        Args:
            team_a_latent: (batch_size, latent_dim)
            team_b_latent: (batch_size, latent_dim)
        Returns:
            prob: (batch_size, 1) - probability team A wins
        """
        # Concatenate team embeddings
        combined = torch.cat([team_a_latent, team_b_latent], dim=1)
        
        # Forward pass
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        prob = self.sigmoid(x)
        
        return prob


class RecurrentGamePredictor(nn.Module):
    """Complete model: team encoders + prediction head."""
    
    def __init__(self, input_size, hidden_dim, num_layers, latent_dim, dropout=0.3):
        super(RecurrentGamePredictor, self).__init__()
        
        # Shared team encoder
        self.encoder = TeamEncoder(
            input_size=input_size,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            latent_dim=latent_dim,
            dropout=dropout
        )
        
        # Prediction head
        self.head = PredictionHead(latent_dim, hidden_dim=hidden_dim)
        
    def forward(self, team_a_seq, team_b_seq):
        """
        Args:
            team_a_seq: (batch_size, seq_length, input_size)
            team_b_seq: (batch_size, seq_length, input_size)
        Returns:
            prob: (batch_size, 1) - probability team A wins
        """
        # Encode teams
        team_a_latent = self.encoder(team_a_seq)
        team_b_latent = self.encoder(team_b_seq)
        
        # Predict
        prob = self.head(team_a_latent, team_b_latent)
        
        return prob, team_a_latent, team_b_latent


print("✓ Model architecture defined")
print(f"  TeamEncoder: LSTM({CONFIG['hidden_dim']}) -> Linear({CONFIG['latent_dim']})")
print(f"  PredictionHead: Concat(2×{CONFIG['latent_dim']}) -> Binary")

✓ Model architecture defined
  TeamEncoder: LSTM(64) -> Linear(32)
  PredictionHead: Concat(2×32) -> Binary


## 3. Data Preparation for Walk-Forward Training

In [5]:
## 3. Data Preparation and Feature Engineering

class GameSequenceDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for NCAA basketball games with lazy loading.
    
    For each game, returns:
    - team_1_seq: (n_prev_games1, n_features) - Team 1's ALL previous games
    - team_2_seq: (n_prev_games2, n_features) - Team 2's ALL previous games
    - outcome: 0 or 1 (Team 1 win?)
    
    Feature engineering happens at init to create model-ready inputs.
    Uses variable-length sequences (all available history).
    """
    
    def __init__(self, df):
        """
        Args:
            df: DataFrame with columns: Team 1, Team 2, Date, Site, Outcome, 
                1-ORtg, 1-DRtg, ... (stats prefixed with 1- and 2-)
        """
        self.df = df.reset_index(drop=True)
        
        # Auto-detect stat columns (all columns starting with '1-')
        self.stat_cols = sorted([col for col in df.columns if col.startswith('1-')])
        self.stat_cols_team2 = [col.replace('1-', '2-') for col in self.stat_cols]

        # Feature engineering at init
        self._prepare_features()
        
        # Build team history lookup
        self._build_team_history()
        


    def __len__(self):
        return len(self.df)
    
    
    def _prepare_features(self):
        """
        - Encode categorical variables (Team names, Site)
        - Normalize ranges
        """
        # 1. Encode Team names
        unique_teams = pd.concat([self.df['Team 1'], self.df['Team 2']]).unique()
        self.team_encoder = {team: idx for idx, team in enumerate(sorted(unique_teams))}
        self.n_teams = len(self.team_encoder)
        self.df['Team 1'] = self.df['Team 1'].map(self.team_encoder)
        self.df['Team 2'] = self.df['Team 2'].map(self.team_encoder)
        
        # 2. Encode Site/Location
        unique_sites = self.df['Site'].unique()
        self.site_encoder = {site: idx for idx, site in enumerate(sorted(unique_sites))}
        
        # 3. Encode Date features
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        # sort by date to ensure chronological order
        self.df = self.df.sort_values(by='Date').reset_index(drop=True)
        
        # 4. Normalization
        
    
    def _build_team_history(self):
        """Build lookup: team_name -> sorted list of game indices"""
        self.team_history = {}
        
        for idx, row in self.df.iterrows():
            team_1 = row['Team 1']
            team_2 = row['Team 2']
            
            if team_1 not in self.team_history:
                self.team_history[team_1] = []
            if team_2 not in self.team_history:
                self.team_history[team_2] = []
            
            self.team_history[team_1].append(idx)
            self.team_history[team_2].append(idx)
    

    def _get_team_sequence(self, team_name, current_game_idx):
        """
        Get ALL previous games for a team before current_game_idx.
        
        Args:
            team_name: Team identifier
            current_game_idx: Index of current game (exclude this and later)
        
        Returns:
            np.array: (n_prev_games, n_features) - sequence of all historical stats
                      Returns zeros if no previous games available
        """
        if team_name not in self.team_history:
            return np.zeros((0, len(self.stat_cols)), dtype=np.float32)
        
        # Get all previous games
        team_game_indices = [idx for idx in self.team_history[team_name] 
                            if idx < current_game_idx]
        
        if len(team_game_indices) == 0:
            return np.zeros((0, len(self.stat_cols)), dtype=np.float32)
        
        # Extract stats from ALL previous games (in chronological order)
        game_seqs = []
        for game_idx in team_game_indices:
            game_row = self.df.loc[game_idx]
            
            # Determine which team slot this team was in
            if game_row['Team 1'] == team_name:
                stats = game_row[self.stat_cols].values.astype(np.float32)
            else:
                stats = game_row[self.stat_cols_team2].values.astype(np.float32)
            
            game_seqs.append(stats)
        
        game_seqs = np.array(game_seqs, dtype=np.float32)
        
        return game_seqs
    

    def __getitem__(self, idx):
        """
        Returns:
            (team_1_seq, team_2_seq, outcome)
            - team_1_seq: (n_prev_games_1, n_features) - ALL Team 1 history
            - team_2_seq: (n_prev_games_2, n_features) - ALL Team 2 history
            - outcome: 0 or 1
            
        Note: Sequences are variable length! Use collate_fn for batching.
        """
        row = self.df.loc[idx]
        
        team_1 = row['Team 1']
        team_2 = row['Team 2']
        outcome = int(row['Outcome'])
        
        # Get ALL historical sequences (all games before this one)
        team_1_seq = self._get_team_sequence(team_1, idx)
        team_2_seq = self._get_team_sequence(team_2, idx)
        
        return (
            torch.FloatTensor(team_1_seq),
            torch.FloatTensor(team_2_seq),
            torch.LongTensor([outcome])
        )


# Custom collate function for variable-length sequences
def collate_variable_length(batch):
    """
    Collate variable-length sequences.
    Pads shorter sequences to match the longest in the batch.
    """
    team_a_seqs, team_b_seqs, outcomes = zip(*batch)
    # Find max lengths in this batch
    max_len_a = max(seq.shape[0] for seq in team_a_seqs) 
    max_len_b = max(seq.shape[0] for seq in team_b_seqs)
    
    n_features = team_a_seqs[0].shape[1]
    batch_size = len(batch)
    
    # Pad sequences
    team_a_padded = torch.zeros(batch_size, max_len_a, n_features)
    team_b_padded = torch.zeros(batch_size, max_len_b, n_features)
    for i, (seq_a, seq_b) in enumerate(zip(team_a_seqs, team_b_seqs)):
        if seq_a.shape[0] > 0:
            team_a_padded[i, :seq_a.shape[0]] = seq_a
        if seq_b.shape[0] > 0:
            team_b_padded[i, :seq_b.shape[0]] = seq_b
    
    outcomes_stacked = torch.cat(outcomes, dim=0)
    
    return team_a_padded, team_b_padded, outcomes_stacked



## 5. Walk-Forward Training

In [6]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for team_a_seq, team_b_seq, targets in train_loader:
        team_a_seq = team_a_seq.to(device)
        team_b_seq = team_b_seq.to(device)
        targets = targets.to(device).view(-1, 1)
        
        optimizer.zero_grad()
        
        # Forward
        prob, _, _ = model(team_a_seq, team_b_seq)
        loss = criterion(prob, targets.float())
        
        # Backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def evaluate(model, val_loader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for team_a_seq, team_b_seq, targets in val_loader:
            team_a_seq = team_a_seq.to(device)
            team_b_seq = team_b_seq.to(device)
            targets = targets.to(device).view(-1, 1)
            
            prob, _, _ = model(team_a_seq, team_b_seq)
            loss = criterion(prob, targets.float())
            
            total_loss += loss.item()
            all_probs.append(prob.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    
    return total_loss / len(val_loader), all_probs, all_targets


print("✓ Training functions defined")

✓ Training functions defined


## 6. Execute Walk-Forward Validation

In [43]:
train_df = df[df['is_tournament_game'] == False].copy()
test_df = df[df['is_tournament_game'] == True].copy()


train_dataset = GameSequenceDataset(train_df)
test_dataset = GameSequenceDataset(test_df)

model = RecurrentGamePredictor(
    input_size=len(train_dataset.stat_cols),
    hidden_dim=CONFIG['hidden_dim'],
    num_layers=CONFIG['num_layers'],
    latent_dim=CONFIG['latent_dim'],
    dropout=CONFIG['dropout']
).to(device)

# Setup optimizer and loss
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)
criterion = nn.BCELoss()

# Create DataLoader
train_loader = DataLoader(
    train_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True,
    collate_fn=collate_variable_length
)

for epoch in tqdm(range(CONFIG['epochs'])):
    model.train()
    total_loss = 0
    
    for team_a_seq, team_b_seq, targets in train_loader:
        # Move to device
        team_a_seq = team_a_seq.to(device)
        team_b_seq = team_b_seq.to(device)
        targets = targets.to(device).view(-1, 1).float()
        
        # Forward pass
        optimizer.zero_grad()
        prob, _, _ = model(team_a_seq, team_b_seq)
        loss = criterion(prob, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
         
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {avg_loss:.4f}")

print("\n✓ Training completed!")

KeyError: 'is_tournament_game'

In [41]:
# Validation
model.eval()
test_loader = DataLoader(
    test_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False,
    collate_fn=collate_variable_length
)

all_probs = []
all_targets = []

with torch.no_grad():
    for team_a_seq, team_b_seq, targets in test_loader:
        team_a_seq = team_a_seq.to(device)
        team_b_seq = team_b_seq.to(device)
        targets = targets.to(device).view(-1, 1)
        
        prob, _, _ = model(team_a_seq, team_b_seq)
        print(prob, targets)
        all_probs.append(prob.cpu().numpy())
        all_targets.append(targets.cpu().numpy())   

all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)

auc = roc_auc_score(all_targets, all_probs)
print(f"✓ Validation AUC: {auc:.4f}")

AttributeError: 'LogisticRegression' object has no attribute 'eval'